# 🚀 V2 Pipeline Orchestrator

Runs all schema and data loading notebooks in the correct FK dependency order.

| Phase | Step | Notebook | What It Does |
|-------|------|----------|--------------|
| Schema | 1 | `model_date` | Creates `shared.DimDate` table |
| Schema | 2 | `model_product` | Creates ProductLine, ProductCategory, Product |
| Schema | 3 | `model_supplychain` | Creates Suppliers, ProductSuppliers, Events, EventImpacts |
| Schema | 4 | `model_inventory` | Creates Warehouses, Inventory, Txns, POs, Forecast |
| Load | 5 | `load_date` | Populates DimDate dynamically (2018 → today + 6 months) |
| Load | 6 | `load_product` | Loads product dimension data |
| Load | 7 | `load_supplychain` | Loads supplier and event data |
| Load | 8 | `load_inventory` | Loads inventory, PO, and forecast data |

**Safe to re-run** — all load notebooks use `MERGE INTO` (upsert).

## Phase 1 — Create Schemas and Tables

### Step 1/8 — Creating date dimension schema

In [ ]:
%run model_date

### Step 2/8 — Creating product schema

In [ ]:
%run model_product

### Step 3/8 — Creating supplychain schema

In [ ]:
%run model_supplychain

### Step 4/8 — Creating inventory schema

In [ ]:
%run model_inventory

## Phase 2 — Load Data

### Step 5/8 — Loading date dimension

In [ ]:
%run load_date

### Step 6/8 — Loading product data

In [ ]:
%run load_product

### Step 7/8 — Loading supplychain data

In [ ]:
%run load_supplychain

### Step 8/8 — Loading inventory data

In [ ]:
%run load_inventory

## Phase 3 — Validation Summary

In [ ]:
from datetime import datetime

print("="*60)
print("📊 FINAL VALIDATION")
print("="*60)

schemas = {
    "shared": ["DimDate"],
    "product": ["ProductLine", "ProductCategory", "Product"],
    "supplychain": ["Suppliers", "ProductSuppliers", "SupplyChainEvents", "SupplyChainEventImpacts"],
    "inventory": ["Warehouses", "Inventory", "InventoryTransactions", "PurchaseOrders", "PurchaseOrderItems", "DemandForecast"]
}

total_tables = 0
total_rows = 0
all_ok = True

for schema, tables in schemas.items():
    print(f"\n  Schema: {schema}")
    for table in tables:
        try:
            count = spark.sql(f"SELECT COUNT(*) as cnt FROM {schema}.{table}").first()["cnt"]
            status = "✅" if count > 0 else "⚠️ EMPTY"
            if count == 0:
                all_ok = False
            print(f"    {status} {schema}.{table}: {count:,} rows")
            total_tables += 1
            total_rows += count
        except Exception as e:
            print(f"    ❌ {schema}.{table}: {str(e)[:80]}")
            all_ok = False

print(f"\n{'='*60}")
if all_ok:
    print(f"🎉 ALL GOOD! {total_tables} tables, {total_rows:,} total rows")
else:
    print(f"⚠️  ISSUES FOUND — check empty/failed tables above")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")